# Historical Channel Extraction and Rice Calendar Preparation

## Purpose and status

This notebook preserves the exploratory research workflow used to extract candidate environmental channels, assemble wheat tables, and prepare rice-calendar variables.
It is annotated as historical source material and was not rerun during cleanup.
The code cells retain their original calculations and order, including hard-coded local paths, Earth Engine exports, and known historical quirks.

The supported replication workflow is the Python package under `replication/`.
For the confirmed manuscript channel scope, `python -m replication.extract_channels` exports only four-week ESI and Aqua-selected FPAR statistics.
The broader notebook branches also calculate ET, root-zone soil moisture, Terra FPAR, and PM2.5, but those are historical experiments rather than the canonical channel selection.

## Historical methods represented here

- **Four-week ESI:** monthly mean, minimum, and maximum from `projects/climate-engine/esi/4wk`.
- **FPAR:** monthly mean, minimum, and maximum from `MODIS/061/MCD15A3H`, scaled by `0.01`.
  The Aqua branch keeps pixels where `FparLai_QC` bits 5-7 are not 4 and sensor bit 1 equals 1.
- **Additional historical candidates:** USGS SSEBop ET, SMAP L4 root-zone soil moisture, Terra-selected FPAR, and CAMS PM2.5.
- **Crop mask:** private JAXA Vietnam land-cover asset, band `b1`, with rice class 3.
- **Rice calendar:** NetCDF harvest group and cropping variables converted to GeoTIFF, sampled at coordinate points, and converted from day of year to month.

## Dependencies and access

The historical environment imports Earth Engine, geemap, pandas, NumPy, Rasterio, rasterstats, rioxarray, GeoPandas, GDAL, xarray, Dask, tqdm, and the local `src/gee_s1_ard` and `src.dhelper` modules.
Running the Earth Engine cells requires access to the private crop-mask asset and permission to create Google Drive exports.
Several cells reference machine-specific `D:` and `G:` paths and a local Downloads folder; update those paths only as an explicit new run configuration.
The setup cell automatically starts Earth Engine authentication when initialization fails, so review it before execution.

## Inputs and outputs

Inputs include `JAXA_subtiles.json`, private Earth Engine assets, exported rice and wheat rasters or CSVs, two rice-calendar NetCDF files, and north/south coordinate tables.
Outputs include Earth Engine Drive CSV or GeoTIFF tasks, zonal-statistics CSVs, merged wheat CSVs, six calendar GeoTIFFs, and sampled harvest date/month CSVs.
The exact paths and Drive folders remain visible in the code cells below.

## Reproducibility notes

- Earth Engine `filterDate` uses an exclusive end date.
  The notebook supplies the last calendar day, so that day is omitted from each historical monthly window.
- The broad channel stack is sampled jointly.
  Missing ET, soil-moisture, or Terra-FPAR pixels can therefore remove rows that would be valid for the canonical ESI and Aqua-FPAR stack.
- The calendar conversion passes coordinate centers to `from_bounds` as outer bounds.
  Prior validation found ascending longitude and descending latitude in the source NetCDF files, so the historical array rows require no latitude flip.
- The DOY conversion overwrites the sampled `*_doy` columns with 2019 month numbers before writing the file named `rice_marc_harvest_dates.csv`.
  The later output renames those columns to `*_moy` and keeps only cropping columns.
- Saved outputs and execution counts were cleared during cleanup because they were stale, noisy, or dependent on unavailable private inputs.
  No cloud task, local export, download, or model computation was executed.

## Source references

- [Evaporative Stress Index overview](https://www.drought.gov/data-maps-tools/evaporative-stress-index-esi)
- [ECOSTRESS ESI algorithm document](https://lpdaac.usgs.gov/documents/340/ECO4ESIALEXIU_ATBD_V1.pdf)
- [SMAP L4 soil-moisture catalog](https://developers.google.com/earth-engine/datasets/catalog/NASA_SMAP_SPL4SMGP_007)
- [USGS MODIS SSEBop ET catalog](https://gee-community-catalog.org/projects/usgs_modis_et/)


## 1. Historical environment setup

### Original cell 3 - Imports, local modules, and Earth Engine initialization

Loads the broad exploratory environment and authenticates Earth Engine if initialization fails. This cell has side effects and depends on the original Google Drive checkout.

In [ ]:
import os
import glob
import json
import sys
import time

import pandas as pd
pd.set_option('display.max_columns', 500)
import numpy as np
import rasterio
import rasterstats
import rioxarray as rxr
import geopandas as gpd
import geemap
from osgeo import gdal
import xarray as xr

dir = 'G:/My Drive/Work/ADB/SAR/'
sys.path.append('G:/My Drive/Work/ADB/SAR/src/gee_s1_ard/python-api')
import border_noise_correction as bnc
import speckle_filter as sf
import terrain_flattening as trf
from src.dhelper import get_month_range, get_last_day_of_month_from_Ym, get_prev_ndays

import ee
# Trigger the authentication flow.
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

### Original cell 4 - Global extraction parameters

Defines CRS, scale, the 2017-01 through 2022-12 month range, and JAXA subtile bounds. Commented wheat and maize branches are retained as historical alternatives.

In [ ]:
crs = 'EPSG:4326'
# scale= 100
scale=100

START_YM = '2017-01'
END_YM = '2022-12'
months = get_month_range(START_YM, END_YM, input_format='%Y-%m', output_format='%Y-%m')

with open("D:\Work\ADB\SAR\shapefiles\JAXA_subtiles.json", 'r') as json_file:
    jaxa_subtiles = json.load(json_file)

# # Wheat tiles 
# chosen_tiles = ['N10E105_0_1', 'N10E105_1_1', 'N10E106_0_1', 'N10E105_1_0','N09E105_1_0', 'N10E104_1_1', 'N09E105_0_0', 'N09E105_1_0', 'N09E105_0_1', 'N09E105_1_1', 'N10E105_0_0', 'N10E106_0_0']
# # Maize tile 
# chosen_tiles = ['N09E106_0_0', 'N09E105_0_0', 'N09E105_0_0', 'N10E105_0_1', 'N10E105_0_0', 'N10E105_1_0',
#                 'N10E106_0_0', 'N10E106_1_0', 'N10E106_1_1']   


# # Wheat Calendar Mask
# wheat = ee.Image('projects/ee-sonle96/assets/worldcereal_wintercereals_vn')
# wheat_mask = wheat.select('b1').eq(100)

# # Maize Calendar Mask
# maize = ee.Image('projects/ee-sonle96/assets/worldcereal_maize_vn')
# maize_mask = maize.select('b1').eq(100)

# crop_type = 'Rice'

# if crop_type == 'Wheat':
#     def mask_crop_map(image):
#         return image.updateMask(wheat_mask)

# if crop_type == 'Maize':
#     def mask_crop_map(image):
#         return image.updateMask(maize_mask)


## 2. Historical all-channel Earth Engine export

### Original cell 5 - Monthly tile CSV exports

Builds the broad candidate stack for the first 16 JAXA subtiles and every configured month, then starts Drive CSV exports. The stack includes ESI, ET, SMAP root-zone soil moisture, and Terra/Aqua FPAR. The canonical replication path keeps only ESI and Aqua-selected FPAR, with the Aqua QC rule shown here. Because `sample` uses the joint stack mask, this historical broad stack can retain fewer rows than the canonical stack.

In [ ]:
print(f"CRS: {crs}")
print(f"scale: {scale}")
print(f"Start: {START_YM}. End: {END_YM}")

# for tile in chosen_tiles[:1]:
for tile in list(jaxa_subtiles.keys())[:16]:
# for tile in list(jaxa_subtiles.keys())[33:]:
    print(f"  {tile}")
    bbox = jaxa_subtiles[tile]
    ROI = ee.Geometry.BBox(bbox[0],
                            bbox[1],
                            bbox[2],
                            bbox[3])
    
    # JAXA LULC VN dataset
    # https://www.eorc.jaxa.jp/ALOS/en/dataset/lulc/lulc_vnm_v2309_e.htm
    jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(ROI).reproject('EPSG:4326', None, 10)
    mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class

    def mask_crop_map(image):
        return image.updateMask(mask_jaxa)

    for month in months[:]:
        print(f"    {month}")

        START_DATE = month + '-01'
        STOP_DATE = get_last_day_of_month_from_Ym(month,
                                                input_format='%Y-%m',
                                                output_format='%Y-%m-%d')
        
        esi_4wk_ic = ee.ImageCollection('projects/climate-engine/esi/4wk')\
                       .filterDate(START_DATE, STOP_DATE)\
                       .map(mask_crop_map)
        
        esi_4wk_mean = esi_4wk_ic.select('ESI')\
                                 .mean()\
                                 .rename('ESI_4wk_mean')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)\
                                #  .updateMask(wheat_mask)
        
        esi_4wk_min = esi_4wk_ic.select('ESI')\
                                 .min()\
                                 .rename('ESI_4wk_min')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)

        esi_4wk_max = esi_4wk_ic.select('ESI')\
                                 .max()\
                                 .rename('ESI_4wk_max')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)
        
        
        modis_et_d_ic = ee.ImageCollection('projects/earthengine-legacy/assets/projects/usgs-ssebop/modis_et_v5_dekadal')\
                          .filterDate(START_DATE, STOP_DATE)\
                          .map(mask_crop_map)
        
        et_mean = modis_et_d_ic.select('et')\
                               .mean()\
                               .rename('ET_mean')\
                               .reproject(crs=crs, scale=scale)\
                               .clip(ROI)
        
        et_min = modis_et_d_ic.select('et')\
                               .min()\
                               .rename('ET_min')\
                               .reproject(crs=crs, scale=scale)\
                               .clip(ROI)
        
        et_max = modis_et_d_ic.select('et')\
                               .max()\
                               .rename('ET_max')\
                               .reproject(crs=crs, scale=scale)\
                               .clip(ROI)
        
        smap_l4_ic = ee.ImageCollection('NASA/SMAP/SPL4SMGP/007')\
                       .filterDate(START_DATE, STOP_DATE)\
                       .map(mask_crop_map)
        
        sm_root_mean = smap_l4_ic.select('sm_rootzone')\
                                 .mean()\
                                 .rename('SM_ROOT_mean')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)
        
        sm_root_min = smap_l4_ic.select('sm_rootzone')\
                                 .min()\
                                 .rename('SM_ROOT_min')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)
        
        sm_root_max = smap_l4_ic.select('sm_rootzone')\
                                 .max()\
                                 .rename('SM_ROOT_max')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)
        
        
        MCD15A3H = ee.ImageCollection("MODIS/061/MCD15A3H")\
                     .filterDate(START_DATE, STOP_DATE)\
                     .map(mask_crop_map)

        def extract_bits(image, bit_start, bit_end):
            num_bits = bit_end - bit_start + 1
            mask = (1 << num_bits) - 1
            return image.rightShift(bit_start).bitwiseAnd(mask)

        def apply_bitmask_terra(image):

            qc_band = image.select('FparLai_QC')

            # Extract values for the specific bit ranges
            # modland_qc = extract_bits(qc_band, 0, 0).eq(0)  # Bit 0 should be 0 (Good quality - main algorithm with or without saturation)
            # detector = extract_bits(qc_band, 2, 2).eq(0)    # Bit 2 should be 0
            # cloud_state = extract_bits(qc_band, 3, 4).eq(0) # Bits 3-4 should be 0 (clear)
            scf_qc = extract_bits(qc_band, 5, 7).neq(4)     # Bits 5-7 should not be 4 (Pixel not produced at all, value couldn''t be retrieved)

            # combined_mask = modland_qc.And(detector).And(cloud_state).And(scf_qc)

            # Sensor
            terra_mask = scf_qc.And(extract_bits(qc_band, 1, 1).eq(0)) 

            return image.updateMask(terra_mask)

        def apply_bitmask_aqua(image):

            qc_band = image.select('FparLai_QC')

            # Extract values for the specific bit ranges
            # modland_qc = extract_bits(qc_band, 0, 0).eq(0)  # Bit 0 should be 0 (Good quality - main algorithm with or without saturation)
            # detector = extract_bits(qc_band, 2, 2).eq(0)    # Bit 2 should be 0
            # cloud_state = extract_bits(qc_band, 3, 4).eq(0) # Bits 3-4 should be 0 (clear)
            scf_qc = extract_bits(qc_band, 5, 7).neq(4)     # Bits 5-7 should not be 4 (Pixel not produced at all, value couldn''t be retrieved)

            # combined_mask = modland_qc.And(detector).And(cloud_state).And(scf_qc)
            # aqua_mask =  combined_mask.And(extract_bits(qc_band, 1, 1).eq(1)) 
            aqua_mask =  scf_qc.And(extract_bits(qc_band, 1, 1).eq(1)) 

            return image.updateMask(aqua_mask)

        dataset_terra = MCD15A3H.map(apply_bitmask_terra)

        fpar_terra_mean = dataset_terra.select('Fpar')\
                                       .mean()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Terra_mean')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)
        
        fpar_terra_min = dataset_terra.select('Fpar')\
                                       .min()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Terra_min')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)
        
        fpar_terra_max = dataset_terra.select('Fpar')\
                                       .max()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Terra_max')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)

        dataset_aqua = MCD15A3H.map(apply_bitmask_aqua)

        fpar_aqua_mean = dataset_aqua.select('Fpar')\
                                       .mean()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Aqua_mean')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)
        
        fpar_aqua_min = dataset_aqua.select('Fpar')\
                                       .min()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Aqua_min')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)
        
        fpar_aqua_max = dataset_aqua.select('Fpar')\
                                       .max()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Aqua_max')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)
        
        combined = esi_4wk_mean.addBands(esi_4wk_min)\
                               .addBands(esi_4wk_max)\
                               .addBands(fpar_terra_mean)\
                               .addBands(fpar_terra_min)\
                               .addBands(fpar_terra_max)\
                               .addBands(fpar_aqua_mean)\
                               .addBands(fpar_aqua_min)\
                               .addBands(fpar_aqua_max)\
                               .addBands(sm_root_mean)\
                               .addBands(sm_root_min)\
                               .addBands(sm_root_max)\
                               .addBands(et_mean)\
                               .addBands(et_min)\
                               .addBands(et_max)\

        
        feat_coll = combined.sample(factor=1,
                                    region=ROI,
                                    geometries=True,
                                    scale=scale,
                                    projection=crs)

        def extract_properties(feature):
            properties = {
                'Timestamp': START_DATE,
                'lon': feature.geometry().coordinates().get(0),
                'lat': feature.geometry().coordinates().get(1),
                'ESI_4wk_mean': feature.get('ESI_4wk_mean'),
                'ESI_4wk_min': feature.get('ESI_4wk_min'),
                'ESI_4wk_max': feature.get('ESI_4wk_max'),
                'ET_mean': feature.get('ET_mean'),
                'ET_min': feature.get('ET_min'),
                'ET_max': feature.get('ET_max'),
                'SM_ROOT_mean': feature.get('SM_ROOT_mean'),
                'SM_ROOT_min': feature.get('SM_ROOT_min'),
                'SM_ROOT_max': feature.get('SM_ROOT_max'),
                'FPAR_Terra_mean': feature.get('FPAR_Terra_mean'),
                'FPAR_Terra_min': feature.get('FPAR_Terra_min'),
                'FPAR_Terra_max': feature.get('FPAR_Terra_max'),
                'FPAR_Aqua_mean': feature.get('FPAR_Aqua_mean'),
                'FPAR_Aqua_min': feature.get('FPAR_Aqua_min'),
                'FPAR_Aqua_max': feature.get('FPAR_Aqua_max')
            }
            return ee.Feature(None, properties)

        processed_fc = feat_coll.map(extract_properties)

        bands = ['Timestamp', 'lon', 'lat', 'ESI_4wk_mean', 'ESI_4wk_min', 'ESI_4wk_max',
                'ET_mean', 'ET_min', 'ET_max', 'SM_ROOT_mean', 'SM_ROOT_min', 'SM_ROOT_max',
                'FPAR_Terra_mean', 'FPAR_Terra_min', 'FPAR_Terra_max',
                'FPAR_Aqua_mean', 'FPAR_Aqua_min', 'FPAR_Aqua_max']
        

        task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                            description=f'{tile}_{scale}m_{START_DATE}_rice',
                                            # description='test',
                                            fileFormat='CSV',
                                            selectors=bands,
                                            folder='rice_channels_south_cams')

        task.start()                   

## 3. Local raster inspection and zonal statistics

### Original cell 7 - Inspect one exported raster

Displays raster index 65 from `paths`. This is an out-of-order inspection cell because `paths` is assigned in original cell 8; it is retained to preserve the exploratory record.

In [ ]:
import rioxarray as rxr
ds = rxr.open_rasterio(paths[65])
ds

### Original cell 8 - Build pixel polygons and discover exports

Creates one polygon per valid pixel in the CAMS reference raster and discovers sorted rice raster paths for later zonal statistics.

In [ ]:
import rasterio
from rasterio.features import shapes
import geopandas as gpd
import numpy as np
from shapely.geometry import box

# Load the raster
raster_path = r"D:\Work\ADB\SAR\datasets\rice_south_rasters\South_VN_CAMS.tif"
with rasterio.open(raster_path) as src:
    image = src.read(1)
    transform = src.transform
    crs = src.crs

# Create bounding boxes (polygons) for all non-nodata pixels
mask = image != src.nodata
rows, cols = np.where(mask)

# Generate bounding boxes
bboxes = []
for row, col in zip(rows, cols):
    x_min, y_max = transform * (col, row)
    x_max, y_min = transform * (col + 1, row + 1)
    bboxes.append(box(x_min, y_min, x_max, y_max))

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(geometry=bboxes, crs=crs)
gdf['pixel'] = gdf.index + 1


import glob

paths = glob.glob(r"D:\Work\ADB\SAR\datasets\rice_south_rasters\*rice*")
paths.sort()

### Original cell 9 - Compute per-pixel zonal statistics

Samples 12 bands across all discovered rasters and writes `zonal_stats.csv`. ET band calculations remain commented out exactly as in the historical source.

In [ ]:
from rasterstats import zonal_stats 
import pandas as pd
from pathlib import Path
from tqdm import tqdm_notebook

dfs = []

# ('ESI_4wk_mean', 'ESI_4wk_min', 'ESI_4wk_max', 'FPAR_Terra_mean', 'FPAR_Terra_min', 'FPAR_Terra_max', 'FPAR_Aqua_mean', 'FPAR_Aqua_min', 'FPAR_Aqua_max', 'SM_ROOT_mean', 'SM_ROOT_min', 'SM_ROOT_max', 'ET_mean', 'ET_min', 'ET_max')
for i in tqdm_notebook(range(len(paths))):
    df_poly = gdf.copy(deep=True)
    df_poly['date'] = Path(paths[i]).name[9:16] + '-01'
    df_poly['ESI_4wk_mean'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=1, stats='mean'))
    df_poly['ESI_4wk_min'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=2, stats='min'))
    df_poly['ESI_4wk_max'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=3, stats='max'))

    df_poly['FPAR_Terra_mean'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=4, stats='mean'))
    df_poly['FPAR_Terra_min'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=5, stats='min'))
    df_poly['FPAR_Terra_max'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=6, stats='max'))

    df_poly['FPAR_Aqua_mean'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=7, stats='mean'))
    df_poly['FPAR_Aqua_min'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=8, stats='min'))
    df_poly['FPAR_Aqua_max'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=9, stats='max'))

    df_poly['SM_ROOT_mean'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=10, stats='mean'))
    df_poly['SM_ROOT_min'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=11, stats='min'))
    df_poly['SM_ROOT_max'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=12, stats='max'))

    # df_poly['ET_mean'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=13, stats='mean'))
    # df_poly['ET_min'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=14, stats='min'))
    # df_poly['ET_max'] = pd.DataFrame(zonal_stats(gdf, paths[i], band=15, stats='max'))

    dfs.append(df_poly)

df = pd.concat(dfs, axis=0).reset_index(drop=True)
df.to_csv(r"D:\Work\ADB\SAR\datasets\rice_south_rasters/zonal_stats.csv", index=False)

### Original cell 10 - Inspect one raster path

Displays a selected input filename as a small exploratory check. Its saved output was cleared.

In [ ]:
paths[64]

## 4. Historical regional raster exports

### Original cell 12 - Export one broad-channel GeoTIFF

Builds the same ESI, ET, soil-moisture, and Terra/Aqua FPAR stack over a manually defined southern Vietnam bounding box for the first month, then starts a 500 m Drive GeoTIFF export.

In [ ]:
print(f"CRS: {crs}")
print(f"scale: {scale}")
print(f"Start: {START_YM}. End: {END_YM}")

# 1. Use only one manually defined bbox
bbox = [104.0, 9.0, 107.0, 12.0]  # Change this to your area of interest
tile = 'South_VN'              # Arbitrary label
ROI = ee.Geometry.BBox(*bbox)
scale = 500
crs = 'EPSG:4326'

# 2. Keep the rest of your processing as is, until the combined image is created
jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(ROI).reproject('EPSG:4326', None, 10)
mask_jaxa = jaxa.select('b1').eq(3)  # Class 3 = rice

def mask_crop_map(image):
    return image.updateMask(mask_jaxa)

for month in months[:1]:
    print(f"    {month}")

    START_DATE = month + '-01'
    STOP_DATE = get_last_day_of_month_from_Ym(month,
                                              input_format='%Y-%m',
                                              output_format='%Y-%m-%d')

        
    esi_4wk_ic = ee.ImageCollection('projects/climate-engine/esi/4wk')\
                        .filterDate(START_DATE, STOP_DATE)\
                        .map(mask_crop_map)
            
    esi_4wk_mean = esi_4wk_ic.select('ESI')\
                                 .mean()\
                                 .rename('ESI_4wk_mean')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)\
                                #  .updateMask(wheat_mask)
        
    esi_4wk_min = esi_4wk_ic.select('ESI')\
                                 .min()\
                                 .rename('ESI_4wk_min')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)

    esi_4wk_max = esi_4wk_ic.select('ESI')\
                                 .max()\
                                 .rename('ESI_4wk_max')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)
        
        
    modis_et_d_ic = ee.ImageCollection('projects/earthengine-legacy/assets/projects/usgs-ssebop/modis_et_v5_dekadal')\
                          .filterDate(START_DATE, STOP_DATE)\
                          .map(mask_crop_map)
        
    et_mean = modis_et_d_ic.select('et')\
                               .mean()\
                               .rename('ET_mean')\
                               .reproject(crs=crs, scale=scale)\
                               .clip(ROI)
        
    et_min = modis_et_d_ic.select('et')\
                               .min()\
                               .rename('ET_min')\
                               .reproject(crs=crs, scale=scale)\
                               .clip(ROI)
        
    et_max = modis_et_d_ic.select('et')\
                               .max()\
                               .rename('ET_max')\
                               .reproject(crs=crs, scale=scale)\
                               .clip(ROI)
        
    smap_l4_ic = ee.ImageCollection('NASA/SMAP/SPL4SMGP/007')\
                       .filterDate(START_DATE, STOP_DATE)\
                       .map(mask_crop_map)
        
    sm_root_mean = smap_l4_ic.select('sm_rootzone')\
                                 .mean()\
                                 .rename('SM_ROOT_mean')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)
        
    sm_root_min = smap_l4_ic.select('sm_rootzone')\
                                 .min()\
                                 .rename('SM_ROOT_min')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)
        
    sm_root_max = smap_l4_ic.select('sm_rootzone')\
                                 .max()\
                                 .rename('SM_ROOT_max')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)
        
        
    MCD15A3H = ee.ImageCollection("MODIS/061/MCD15A3H")\
                     .filterDate(START_DATE, STOP_DATE)\
                     .map(mask_crop_map)

    def extract_bits(image, bit_start, bit_end):
            num_bits = bit_end - bit_start + 1
            mask = (1 << num_bits) - 1
            return image.rightShift(bit_start).bitwiseAnd(mask)

    def apply_bitmask_terra(image):

            qc_band = image.select('FparLai_QC')

            # Extract values for the specific bit ranges
            # modland_qc = extract_bits(qc_band, 0, 0).eq(0)  # Bit 0 should be 0 (Good quality - main algorithm with or without saturation)
            # detector = extract_bits(qc_band, 2, 2).eq(0)    # Bit 2 should be 0
            # cloud_state = extract_bits(qc_band, 3, 4).eq(0) # Bits 3-4 should be 0 (clear)
            scf_qc = extract_bits(qc_band, 5, 7).neq(4)     # Bits 5-7 should not be 4 (Pixel not produced at all, value couldn''t be retrieved)

            # combined_mask = modland_qc.And(detector).And(cloud_state).And(scf_qc)

            # Sensor
            terra_mask = scf_qc.And(extract_bits(qc_band, 1, 1).eq(0)) 

            return image.updateMask(terra_mask)

    def apply_bitmask_aqua(image):

            qc_band = image.select('FparLai_QC')

            # Extract values for the specific bit ranges
            # modland_qc = extract_bits(qc_band, 0, 0).eq(0)  # Bit 0 should be 0 (Good quality - main algorithm with or without saturation)
            # detector = extract_bits(qc_band, 2, 2).eq(0)    # Bit 2 should be 0
            # cloud_state = extract_bits(qc_band, 3, 4).eq(0) # Bits 3-4 should be 0 (clear)
            scf_qc = extract_bits(qc_band, 5, 7).neq(4)     # Bits 5-7 should not be 4 (Pixel not produced at all, value couldn''t be retrieved)

            # combined_mask = modland_qc.And(detector).And(cloud_state).And(scf_qc)
            # aqua_mask =  combined_mask.And(extract_bits(qc_band, 1, 1).eq(1)) 
            aqua_mask =  scf_qc.And(extract_bits(qc_band, 1, 1).eq(1)) 

            return image.updateMask(aqua_mask)

    dataset_terra = MCD15A3H.map(apply_bitmask_terra)

    fpar_terra_mean = dataset_terra.select('Fpar')\
                                       .mean()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Terra_mean')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)
        
    fpar_terra_min = dataset_terra.select('Fpar')\
                                       .min()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Terra_min')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)
        
    fpar_terra_max = dataset_terra.select('Fpar')\
                                       .max()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Terra_max')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)

    dataset_aqua = MCD15A3H.map(apply_bitmask_aqua)

    fpar_aqua_mean = dataset_aqua.select('Fpar')\
                                       .mean()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Aqua_mean')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)
        
    fpar_aqua_min = dataset_aqua.select('Fpar')\
                                       .min()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Aqua_min')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)
        
    fpar_aqua_max = dataset_aqua.select('Fpar')\
                                       .max()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Aqua_max')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)

    # Final combined image
    combined = esi_4wk_mean.addBands(esi_4wk_min)\
                           .addBands(esi_4wk_max)\
                           .addBands(fpar_terra_mean)\
                           .addBands(fpar_terra_min)\
                           .addBands(fpar_terra_max)\
                           .addBands(fpar_aqua_mean)\
                           .addBands(fpar_aqua_min)\
                           .addBands(fpar_aqua_max)\
                           .addBands(sm_root_mean)\
                           .addBands(sm_root_min)\
                           .addBands(sm_root_max)\
                           .addBands(et_mean)\
                           .addBands(et_min)\
                           .addBands(et_max).toFloat()

    # 3. Export the combined image as GeoTIFF
    task = ee.batch.Export.image.toDrive(
        image=combined,
        description=f'{tile}_{scale}m_{START_DATE}_rice_raster',
        folder='rice_south_rasters',
        fileNamePrefix=f'{tile}_{month}_rice',
        region=ROI,
        scale=scale,
        crs=crs,
        maxPixels=1e13
    )
    
    task.start()

### Original cell 14 - Export one CAMS PM2.5 GeoTIFF

Computes CAMS NRT PM2.5 mean and maximum over the Vietnam bounding box for the first month and starts a coarse Drive export. This is a separate historical experiment, not part of the canonical ESI and Aqua-FPAR channel command.

In [ ]:


# 1. Use only one manually defined bbox
# bbox = [104.0, 9.0, 107.0, 12.0]  # Change this to your area of interest
bbox = [102.170435826, 8.59975962975, 109.33526981, 23.3520633001] # VN
tile = 'South_VN'              # Arbitrary label
ROI = ee.Geometry.BBox(*bbox)
scale = 500
crs = 'EPSG:4326'

# 2. Keep the rest of your processing as is, until the combined image is created
jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(ROI).reproject('EPSG:4326', None, 10)
mask_jaxa = jaxa.select('b1').eq(3)  # Class 3 = rice


for month in months[:1]:
    print(f"    {month}")

    START_DATE = month + '-01'
    STOP_DATE = get_last_day_of_month_from_Ym(month,
                                              input_format='%Y-%m',
                                              output_format='%Y-%m-%d')

    pm25 = ee.ImageCollection("ECMWF/CAMS/NRT")\
              .filter(ee.Filter.date(START_DATE, STOP_DATE))

    pm25_mean = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                    .mean()\
                    .multiply(1E9)\
                    .rename('PM25_mean')\
                    .clip(ROI)\
                    .reproject(crs=crs)
    
    pm25_max = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                    .max()\
                    .multiply(1E9)\
                    .rename('PM25_max')\
                    .clip(ROI)\
                    .reproject(crs=crs) 
    
    combined = pm25_mean.addBands(pm25_max).toFloat()

    # 3. Export the combined image as GeoTIFF
    task = ee.batch.Export.image.toDrive(
        image=combined,
        description=f'{tile}_CAMS_{START_DATE}_rice_raster',
        folder='rice_south_rasters',
        fileNamePrefix=f'{tile}_{month}_CAMS_VN_whole',
        region=ROI,
        crs=crs,
        scale=44528,
        maxPixels=1e13
    )
    
    task.start()

## 5. Historical wheat-table assembly

### Original cell 17 - Load extra wheat metrics

Concatenates all CSV files from the historical `wheat_extra_metrics` directory. The progress display and table output were cleared.

In [ ]:
from tqdm import tqdm_notebook
import glob
import pandas as pd

dfs = []
wheat_xtra_paths = glob.glob("G:\My Drive\Work\ADB\SAR\wheat_extra_metrics\*")
for path in tqdm_notebook(wheat_xtra_paths):
    df = pd.read_csv(path)
    dfs.append(df)

wheat_extra_df = pd.concat(dfs, axis=0).reset_index(drop=True)
wheat_extra_df

### Original cell 18 - Load the base wheat table

Reads the historical southern wheat CSV for subsequent joins.

In [ ]:
wheat_df = pd.read_csv(r"D:\Work\ADB\SAR\datasets\wheat_south.csv")
wheat_df

### Original cell 19 - Outer-join base and extra metrics

Merges on timestamp and coordinates while retaining unmatched rows from either table.

In [ ]:
merged = pd.merge(wheat_df, wheat_extra_df, on=['Timestamp', 'lon', 'lat'], how='outer')
merged

### Original cell 20 - Define a non-null subset

Keeps rows with ESI mean, ET maximum, and NDVI mean. The direct export remains commented out.

In [ ]:

merged_nn = merged[(merged['ESI_4wk_mean'].notnull()) & (merged['ET_max'].notnull()) & (merged['NDVI_mean'].notnull())]
# merged_nn.to_csv(r'D:\Work\ADB\SAR\datasets/wheat_south_full_nonnull.csv', index=False)

### Original cell 21 - Export the full wheat merge

Writes the outer-joined table to `wheat_south_full.csv`.

In [ ]:
merged.to_csv(r'D:\Work\ADB\SAR\datasets/wheat_south_full.csv', index=False)

### Original cell 22 - Load historical SAR metrics

Concatenates the wheat SAR CSV files for later coordinate-time joins.

In [ ]:
dfs = []
wheat_sar_paths = glob.glob("D:\Work\ADB\SAR\wheat_sar/*")
for path in tqdm_notebook(wheat_sar_paths):
    df = pd.read_csv(path)
    dfs.append(df)

wheat_sar_df = pd.concat(dfs, axis=0).reset_index(drop=True)
wheat_sar_df

### Original cell 23 - Left-join SAR metrics

Adds SAR fields to the full merged table and writes `wheat_south_sar_full.csv`.

In [ ]:
merged_sar = pd.merge(merged, wheat_sar_df, on=['Timestamp', 'lat', 'lon'], how='left')
merged_sar.to_csv(r'D:\Work\ADB\SAR\datasets/wheat_south_sar_full.csv', index=False)


### Original cell 24 - Build an inner non-null SAR merge

Creates `merged_sar_nn`, but the historical export statement writes `merged_sar` to the `nnull` filename. The mismatch is documented here and left unchanged.

In [ ]:
merged_sar_nn = pd.merge(merged_nn, wheat_sar_df, on=['Timestamp', 'lat', 'lon'], how='inner')
merged_sar.to_csv(r'D:\Work\ADB\SAR\datasets/wheat_south_sar_nnull.csv', index=False)

### Original cell 25 - Inspect the full SAR merge

Displays the merged table. The stale saved table was cleared.

In [ ]:
merged_sar

### Original cell 26 - Reload the saved non-null SAR table

Reads the historical `wheat_south_sar_nonnull.csv` artifact and displays it.

In [ ]:
merged_sar_nn = pd.read_csv(r'D:\Work\ADB\SAR\datasets/wheat_south_sar_nonnull.csv')
merged_sar_nn

### Original cell 27 - Inspect complete rows

Displays rows remaining after a full `dropna` operation without modifying the stored dataframe.

In [ ]:
merged_sar_nn.dropna()

## 6. Rice-calendar NetCDF conversion

### Original cell 30 - Convert harvest-group variables to GeoTIFF

Writes three group rasters with EPSG:4326 and nodata `-9999`. The historical transform treats the first and last coordinate centers as raster bounds; prior validation preserved this behavior for reproducibility.

In [ ]:
from rasterio.transform import from_bounds


ds = xr.open_dataset(r"C:\Users\sonle\Downloads\Rice_Calendar_Data_ver.1.0\Rice_Calendar_Data_ver.1.0\1_Transplanting_Harvest_Group.nc")
for var in ['Harvest_1_Group', 'Harvest_2_Group', 'Harvest_3_Group']:
    print(var)
    data = ds[var]

    min_lon, max_lon = data.lon.min().item(), data.lon.max().item()
    min_lat, max_lat = data.lat.min().item(), data.lat.max().item()

    min_lon, max_lon = data.lon.min().item(), data.lon.max().item()
    min_lat, max_lat = data.lat.min().item(), data.lat.max().item()

    transform = from_bounds(min_lon, min_lat, max_lon, max_lat, data.sizes['lon'], data.sizes['lat'])

    output_tiff_path = f'D:/Work/ADB/SAR/calendars/marc_{var.lower()}.tiff'
    meta = {
        'driver': 'GTiff',
        'height': data.shape[0],
        'width': data.shape[1],
        'count': 1,
        'dtype': 'float32',
        'crs': 'EPSG:4326',  # Assuming the data uses WGS84
        'transform': transform,
        'nodata': -9999
    }

    # Write the GeoTIFF file
    with rasterio.open(output_tiff_path, 'w', **meta) as dst:
        dst.write(data, 1)


    

### Original cell 31 - Convert harvest-cropping variables to GeoTIFF

Writes the three cropping rasters using the same historical transform. The source latitude axis is descending, so the historical arrays are already north-up and require no row flip.

In [ ]:
ds = xr.open_dataset(r"C:\Users\sonle\Downloads\Rice_Calendar_Data_ver.1.0\Rice_Calendar_Data_ver.1.0\3_Transplanting_Harvest_Cropping.nc")
for var in ['Harvest_1_Cropping', 'Harvest_2_Cropping', 'Harvest_3_Cropping']:
    print(var)
    data = ds[var]

    min_lon, max_lon = data.lon.min().item(), data.lon.max().item()
    min_lat, max_lat = data.lat.min().item(), data.lat.max().item()

    min_lon, max_lon = data.lon.min().item(), data.lon.max().item()
    min_lat, max_lat = data.lat.min().item(), data.lat.max().item()

    transform = from_bounds(min_lon, min_lat, max_lon, max_lat, data.sizes['lon'], data.sizes['lat'])

    output_tiff_path = f'D:/Work/ADB/SAR/calendars/marc_{var.lower()}.tiff'
    meta = {
        'driver': 'GTiff',
        'height': data.shape[0],
        'width': data.shape[1],
        'count': 1,
        'dtype': 'float32',
        'crs': 'EPSG:4326',  # Assuming the data uses WGS84
        'transform': transform,
        'nodata': -9999
    }

    # Write the GeoTIFF file
    with rasterio.open(output_tiff_path, 'w', **meta) as dst:
        dst.write(data, 1)


ds

## 7. Coordinate preparation

### Original cell 33 - Start a local Dask cluster

Creates eight worker processes with a 2 GB memory limit per worker for coordinate-table preparation.

In [ ]:
from dask.distributed import Client, LocalCluster
cluster = LocalCluster(n_workers=8, memory_limit="2GB", 
                       processes=True)
client = Client(cluster)

### Original cell 34 - Export unique southern coordinates

Reads the large southern CSV with Dask, deduplicates latitude-longitude pairs, and writes one CSV. The historical output directory name contains `VN_NORTH_VER2` even though the filename is `unique_coords_south.csv`; this is preserved.

In [ ]:
import dask.dataframe as dd

df = dd.read_csv("D:\Work\ADB\SAR\datasets\VN_SOUTH_VER2.csv")
df = df[['lat', 'lon']]
df = df.drop_duplicates(subset=['lat', 'lon'])
df.to_csv("D:\Work\ADB\SAR\datasets\VN_NORTH_VER2/unique_coords_south.csv", index=False, single_file=True)

### Original cell 35 - Stop the Dask client

Shuts down the local Dask resources created above.

In [ ]:
client.shutdown()

## 8. Intermediate schema and row-count checks

### Original cell 38 - Inspect dataframe columns

Displays the current dataframe schema. This depends on interactive state and is retained only as an exploratory checkpoint.

In [ ]:
df.columns

### Original cell 39 - Count province lookup rows

Loads `province_by_coords.csv` and displays its historical row count. The saved count was cleared.

In [ ]:
df = pd.read_csv(r"D:\Work\ADB\SAR\datasets\province_by_coords.csv")
len(df)

## 9. Calendar sampling and DOY-to-month conversion

### Original cell 41 - Combine coordinate tables

Concatenates north and south coordinate files and creates point geometries for raster sampling.

In [ ]:
north = pd.read_csv(r"D:\Work\ADB\SAR\datasets\unique_coords_north.csv")
south = pd.read_csv(r"D:\Work\ADB\SAR\datasets\unique_coords_south.csv")

full = pd.concat([north, south], axis=0).reset_index(drop=True)
gdf = gpd.GeoDataFrame(full,
                       geometry=gpd.points_from_xy(full['lon'], full['lat']))

### Original cell 42 - Sample six calendar rasters

Samples three cropping and three group rasters at every point, converts nodata to `-9999`, and appends `*_doy` columns. The historical georeferencing described above controls which source cell each coordinate samples.

In [ ]:
from shapely import Polygon
import rasterio
from rasterio.features import geometry_mask
from rasterio import sample
from concurrent.futures import ProcessPoolExecutor

def sample_raster_value(args):
    """
    Function to sample raster value at a single point.
    """
    raster_path, coord = args
    with rasterio.open(raster_path) as raster:
        value = list(sample.sample_gen(raster, [coord]))[0][0]
    return value

for var in ['harvest_1_cropping', 'harvest_1_group', 'harvest_2_cropping', 'harvest_2_group',
            'harvest_3_cropping', 'harvest_3_group']:
    print(var)
    raster = rasterio.open(rf"D:\Work\ADB\SAR\calendars\marc_{var}.tiff")
    coords = [(point.x, point.y) for point in gdf.geometry]
    raster_values = [x[0] for x in sample.sample_gen(raster, coords)]
    # with ProcessPoolExecutor() as executor:
    #     raster_values = list(executor.map(sample_raster_value, args))
    raster_values = np.nan_to_num(raster_values, nan=-9999)

    gdf[var + '_doy'] = raster_values





### Original cell 43 - Replace nodata with missing values

Drops geometry and converts the `-9999` sentinel to `NaN`.

In [ ]:
df = pd.DataFrame(gdf.drop(['geometry'], axis=1)).replace(-9999, np.nan)

### Original cell 44 - Inspect sampled calendar values

Displays the sampled table before date conversion. The stale saved table was cleared.

In [ ]:
df

### Original cell 45 - Convert DOY values to 2019 month numbers

Applies the conversion in place to every non-coordinate column. Although the helper defaults to 2024, this loop explicitly uses 2019. After this cell, the `*_doy` column names contain month numbers rather than day-of-year values.

In [ ]:
from datetime import datetime, timedelta

def doy_to_month(doy, year=2024):
    """
    Convert day of year (DOY) to the corresponding month of year (MOY).

    Parameters:
        doy (int): Day of the year (1-365 or 1-366 for leap years).
        year (int): Year to consider (to account for leap years).

    Returns:
        int: The month number (1-12).
    """
    # Start of the year
    start_of_year = datetime(year, 1, 1)
    # Add DOY as timedelta
    if not np.isnan(doy):
        target_date = start_of_year + timedelta(days=doy - 1)
        return target_date.month
    else:
        return np.nan

for col in df.drop(['lon', 'lat'], axis=1).columns:
    df[col] = df[col].apply(lambda x: doy_to_month(x, 2019))

### Original cell 46 - Write the historically named dates CSV

Writes `rice_marc_harvest_dates.csv` after the in-place conversion, so the file contains month numbers under `*_doy` headers.

In [ ]:
df.to_csv("D:/Work/ADB/SAR/datasets/rice_marc_harvest_dates.csv", index=False)

### Original cell 47 - Rename converted columns to `*_moy`

Renames all converted calendar columns from day-of-year to month-of-year suffixes.

In [ ]:
for col in df.drop(['lon', 'lat'], axis=1).columns:
    new_name = col.replace('doy', 'moy')
    df = df.rename({col: new_name}, axis=1)

### Original cell 48 - Write cropping-month output

Drops the three group-month columns and writes the final cropping-month CSV.

In [ ]:
df.drop(['harvest_1_group_moy',
         'harvest_2_group_moy',
         'harvest_3_group_moy'], axis=1).to_csv("D:/Work/ADB/SAR/datasets/rice_marc_harvest_months.csv", index=False)